In [ ]:
import tensorflow as tf
import keras
import glob
import os

Check current runtime

In [ ]:
print("Tensorflow version: ",tf.__version__)
print("Available GPUs:", tf.config.list_physical_devices('GPU')) # check for GPU

Tensorflow version:  2.18.0
Available GPUs: []


Download & un the data

In [ ]:
def get_data_extract():
  if "dataset" in os.listdir():
    print("Dataset already exists")
  else:
    print("Downloading the data...")
    !wget -O food-data.zip https://www.kaggle.com/api/v1/datasets/download/trolukovich/food11-image-dataset
    print("Dataset downloaded!")
    print("Extracting data..")
    !mkdir dataset
    !unzip -q food-data.zip -d dataset
    print("Extraction done!")

get_data_extract()

# Dataset

Create Dataset from list of path

In [ ]:
path = glob.glob("dataset/*/*/*.jpg")
label = [i.split(".")[0].split("/")[-2] for i in path]

Image augmentation

In [53]:
image_width, image_height = 224,244 # this may need to be changed based on loaded model

aug = keras.Sequential([
    keras.layers.Resizing(image_width, image_height),
    keras.layers.Rescaling(1./255), # this may need to be changed based on loaded model
])

Dataloader

In [ ]:
def load_image(path, label, aug):
    image = tf.io.read_file(path)
    image = tf.image.decode_jpeg(image, channels=3)
    # image augmentation
    image = aug(image)
    return image, label

In [ ]:
ds = tf.data.Dataset.from_tensor_slices((path, label))
ds=ds.shuffle(buffer_size=len(path), reshuffle_each_iteration=False) # disable reshuffle_each_iteration to prevent data leak
ds = ds.map(load_image).batch(16).prefetch(tf.data.AUTOTUNE)

Display Sample from our custom dataset

In [54]:
print("Sample from the dataset:",next(iter(ds))[1])

Sample from the dataset: tf.Tensor(
[b'Soup' b'Egg' b'Dessert' b'Dessert' b'Dairy product' b'Noodles-Pasta'
 b'Soup' b'Bread'], shape=(8,), dtype=string)


# Model